# Re-annote les vraies parties externes avec notre propre prof

Contexte : un dataset de vraies parties (883k positions, minimax "insane"
-- profondeur FIXE 22, 1,2s/position) a ete recu du projet
`songo-model-stockfish-for-google-collab`. Plutot que de garder ses
labels, on reconvertit les plateaux vers notre propre format (encodage
valide sur 15 000 echantillons, 0 echec -- voir
`songo_ai.dataset.external_import`) et on les ré-annote avec notre prof
(DeepTeacher, adaptatif, budget de temps nettement plus genereux).

**Le cache d'annotations SQLite doit vivre sur Drive** (pas sur le disque
Colab, ephemere) : si la session se coupe, relancer ce notebook reprend
exactement ou ca s'est arrete, sans rien recalculer.

**Important -- lire avant de lancer le volume complet** : mesure locale,
~15-16s/position avec la config profonde (jusqu'a profondeur 40, 60s
plafond). Sur les 883k positions (apres dedup, probablement un peu
moins), meme avec plusieurs coeurs Colab, ca represente un vrai calcul
de plusieurs jours -- commencer par `--limit` sur un sous-ensemble pour
mesurer le debit REEL sur ce Colab precis avant de viser le volume
complet (meme discipline que pour les paliers GCP precedents).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/SongoFish'  # ajuster si besoin
import os
assert os.path.isdir(DRIVE_ROOT), f"Dossier introuvable : {DRIVE_ROOT}"

# Dossier source : uploader dataset_full_matrix_merged_all_colabs/ sur
# Drive avant de lancer ce notebook (train.npz, validation.npz, test.npz,
# dataset_metadata.json)
EXTERNAL_INPUT_DIR = f'{DRIVE_ROOT}/datasets/dataset_full_matrix_merged_all_colabs'
OUT_DIR = f'{DRIVE_ROOT}/datasets/dataset_external_reannotated'
CACHE_DIR = f'{OUT_DIR}/annotation_cache'  # SUR DRIVE : survit aux deconnexions

In [ ]:
!git clone https://github.com/GlennEriss/songo-fish.git /content/songo-fish
%cd /content/songo-fish
!pip install -q -e ".[train]"

In [ ]:
import multiprocessing
print('coeurs disponibles sur cette session Colab:', multiprocessing.cpu_count())

## Etape 1 -- pilote (mesurer le debit reel avant de s'engager)

In [ ]:
!.venv/bin/python apps/trainer/scripts/reannotate_external_dataset.py \
    --input-dir "{EXTERNAL_INPUT_DIR}" \
    --out-dir /content/pilot_reannotated \
    --cache-dir "{CACHE_DIR}" \
    --limit 100 --num-workers 2

## Etape 2 -- volume complet (une fois le debit du pilote juge acceptable)

Retirer `--limit` pour prendre tout. Le cache etant sur Drive, relancer
cette meme cellule apres une deconnexion reprend sans tout refaire.

In [ ]:
!.venv/bin/python apps/trainer/scripts/reannotate_external_dataset.py \
    --input-dir "{EXTERNAL_INPUT_DIR}" \
    --out-dir "{OUT_DIR}" \
    --cache-dir "{CACHE_DIR}" \
    --num-workers 2

## Etape 3 -- fusion avec le dataset existant (dedup globale)

A faire cote local (ou ici, si `dataset_v005_400k` est aussi sur Drive).

In [ ]:
from pathlib import Path
from songo_ai.dataset import merge_releases

manifest = merge_releases(
    [Path(f'{DRIVE_ROOT}/datasets/dataset_v005_400k'), Path(OUT_DIR)],
    Path(f'{DRIVE_ROOT}/datasets/dataset_v006_merged_external'),
    deduplicate=True,
)
print(manifest['total_positions'], manifest['counts_per_split'])